# Suite REF — References: one format for objects, relation entries and frames

A reference designates what a request or a response points at ([CAP-130: references to objects, entries and frames](../../../mbse-basis/docs/capabilities/CAP-130.md#decided)):
`References.OfObject`, an object, by its path (a singleton, an owned object or a
schema object), never a counted one, which nothing names (object identity 40 and 55 in [CAP-118: singletons, owned and counted objects](../../../mbse-basis/docs/capabilities/CAP-118.md#decided));
`References.OfEntry`, a relation entry, by its relation's path and all its links and properties; `References.OfFrame`,
a frame, each symbol's object and each variable's value, and what it witnesses, objects' property values
(`References.OfProperties`) and relation entries, each by its objects' references. `References.of(store, x)`
gives one, a value of the store's, written and read as any value is; `References.resolve(store, reference)` gives what
it designates, or null where the store holds none.

As in Python, with the language's differences (docs/EQUIVALENCE.md): builders and the store are `Proxy` objects typed
loosely, so callbacks are annotated `(t: any)`; a frame is a `Map`, plain mappings are `Map`s, ints `bigint`s.

In [ ]:
import { Builders as B, JSON as Json, Paths, Plain, References } from "@mbse/schemas/Framework";
import { LookupError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, map, raises } from "./support.js";

/** crm.Person, who knows persons since a year; crm.Book, a singleton owning its contacts by key; crm.Note, counted,
 * about a person. */
function crm(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Knows: S.OfRelation, Person: S.OfObject, Holds: S.OfRelation, Book: S.OfObject,
      Note: S.OfObject, About: S.OfRelation })
    .spec([
      (_: any) => _.Knows.name("crm.Knows").links([(t: any) => t.name("knower"), (t: any) => t.name("known")]).properties({
        since: S.OfInt }),
      (_: any) => _.Person.name("crm.Person").ref(true).properties({ name: S.OfStr }).adjacencies([
        (t: any) => t.name("knows").relation([_.Knows]).me("knower"),
        (t: any) => t.relation([_.Knows]).me("known"),
        (t: any) => t.name("book").relation([_.Holds]).me("contact").owner(true),
      ]).adjacencies({ trio: (t: any) => t.relation([S.OfRelation((r: any) => r.links([ // no name of its own
        (t: any) => t.name("a"), (t: any) => t.name("b"), (t: any) => t.name("c")]))]).me("a") }),
      (_: any) => _.Holds.name("crm.Holds").links([(t: any) => t.name("book"), (t: any) => t.name("contact")]).properties({
        key: S.OfStr }).uniques([["book", "key"], ["contact"]]),
      (_: any) => _.Book.name("crm.Book").ref(true).singleton("crm.TheBook").adjacencies({
        contacts: (t: any) => t.relation([_.Holds]).me("book") }),
      (_: any) => _.Note.name("crm.Note").ref(true).properties({ text: S.OfStr }).adjacencies({
        about: (t: any) => t.relation([_.About]).me("note") }),
      (_: any) => _.About.name("crm.About").links([(t: any) => t.name("note"), (t: any) => t.name("person")]),
    ])]))]);
}

const store: any = new B.Store();
store.transaction(crm);
const [Person, Book, Note] = ["crm.Person", "crm.Book", "crm.Note"].map((name) => store.registered(name));
const book = store.singleton("crm.TheBook");
const frames = store.transaction((t: any) => t.uid("people").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
  .symbols({ book: Book, ada: Person, bob: Person, cy: Person }).spec([
    (_: any) => _.book.contacts({ ada: (e: any) => e.contact(_.ada), bob: (e: any) => e.contact(_.bob),
      cy: (e: any) => e.contact(_.cy) }),
    (_: any) => _.ada.name("Ada").knows([(e: any) => e.known(_.bob).since(2020n), (e: any) => e.known(_.cy).since(2021n)]),
    (_: any) => _.bob.name("Bob"),
    (_: any) => _.cy.name("Cy"),
  ])]))]));
const [ada, bob, cy] = ["ada", "bob", "cy"].map((name) => frames[0].get(name));

/** A note about each person, upserted with the key `notes/1` for the frame that reaches it. */
function notes(t: any): any {
  return t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols({ contact: Person })])
    .then([(n: any) => n.upsert((u: any) => u
    .key("notes/1").exists([(p: any) => p.symbols({ contact: Person, note: Note }).spec([
      (_: any) => _.note.text("hi").about([_.contact])])]))]))]);
}

function plain(reference: any): any {
  return Plain.ToPlain(store)(reference.as_schema(), reference);
}

/** A path's plain form: from a singleton through entries, each [i, o, keys]. */
function path(singleton: string, ...steps: [string, string, Record<string, unknown>][]): unknown {
  let nested: unknown = { target: {} };
  for (const [i, o, keys] of [...steps].reverse()) nested = { entry: { i, o, properties: keys, next: nested } };
  return { singleton, next: nested };
}

## REF-01 · Every store holds the reference schemas, its own; References.install, run again, writes nothing

In [ ]:
const OfObject = store.registered("References.OfObject");
const [OfFrame, OfEntry] = [store.registered("References.OfFrame"), store.registered("References.OfEntry")];
assert(equal([...OfObject.properties.keys()], ["path"]) && OfObject.properties.get("path").schema === store.registered("Paths.OfRoot"));
const OfProperties = store.registered("References.OfProperties");
assert(equal([...OfFrame.properties.keys()], ["binds", "variables", "assigns", "entries"]));
assert(OfFrame.properties.get("binds").schema.element === OfObject && OfFrame.properties.get("assigns").schema.element === OfProperties);
assert(OfFrame.properties.get("entries").schema.element === OfEntry && equal([...OfProperties.properties.keys()], ["object", "properties"]));
assert(equal([...OfEntry.properties.keys()], ["relation", "links", "properties"]));
assert(OfEntry.properties.get("relation").schema === store.registered("Paths.OfRoot"));
assert(OfEntry.properties.get("links").schema.element === OfObject);
assert(![OfObject, OfFrame, OfProperties, OfEntry].some((schema) => schema.ref)); // value objects, as a path is
const other: any = new B.Store();
assert(other.registered("References.OfObject") !== OfObject); // each store's its own
const names = new Map((store.names() as string[]).map((name) => [name, store.registered(name)]));
store.transaction(References.install); // each found by its name
assert(store.names().length === names.size && (store.names() as string[]).every((name) => store.registered(name) === names.get(name)));

## REF-02 · An object's reference by its path: a singleton, an owned object, a schema object; each resolved

In [ ]:
{
  assert(equal(plain(References.of(store, book)), map({ path: path("crm.TheBook") })));
  assert(equal(plain(References.of(store, ada)), map({ path: path("crm.TheBook", ["contacts", "book", { key: "ada" }]) })));
  assert(equal(plain(References.of(store, Person)), map({ path: path(".", ["schemas", "", { name: "crm" }],
    ["schemas", "", { name: "Person" }]) })));
  for (const target of [book, ada, Person]) assert(References.resolve(store, References.of(store, target)) === target);
  const built = store.References.OfObject((o: any) => o.path((p: any) => p.singleton("crm.TheBook").next((n: any) => n.entry(
    (e: any) => e.i("contacts").o("book").properties({ key: "bob" }).next((n: any) => n.target((t: any) => t))))));
  assert(References.resolve(store, built) === bob); // written through the builders' fluent calls
}

## REF-03 · A counted object has no reference, nor a frame that binds one: an upsert's, which the executor alone finds again

In [ ]:
const made = store.transaction(notes);
assert(equal(made.map((frame: any) => frame.get("contact")), [ada, bob, cy]));
const note = made[0].get("note");
raises(LookupError, () => References.of(store, note), "a counted 'crm.Note' has no reference: nothing names it");
raises(LookupError, () => References.of(store, made[0]), "symbol 'note' of the frame: a counted 'crm.Note' has no reference");
const again = store.transaction(notes); // run again: the executor finds what it made, which nothing outside it names
assert(again[0].get("note") === note);

## REF-04 · A relation entry by its relation's path and all its links and properties: a keyed relation's, one without properties; refused where it links a counted object

In [ ]:
{
  const held = [...book.contacts][0]; // crm.Holds, keyed by `key`
  assert(equal(plain(References.of(store, held)), map({
    relation: path(".", ["schemas", "", { name: "crm" }], ["schemas", "", { name: "Holds" }]),
    links: { book: { path: path("crm.TheBook") },
      contact: { path: path("crm.TheBook", ["contacts", "book", { key: "ada" }]) } },
    properties: { key: { str: "ada" } } })));
  const knows = [...ada.knows];
  assert(equal(plain(References.of(store, knows[1])).get("properties"), map({ since: { int: 2021n } })));
  store.transaction(() => {
    store.crm.Person(bob, (b: any) => b.knows((a: any) => a.insert((e: any) => e.known(cy)))); // since left unset
  });
  const bare = [...bob.knows][0];
  assert(equal(plain(References.of(store, bare)).get("properties"), new Map()));
  raises(LookupError, () => References.of(store, [...note.about][0]), "a counted 'crm.Note' has no reference"); // crm.About links a note
  for (const entry of [held, knows[0], knows[1], bare]) assert(References.resolve(store, References.of(store, entry)) === entry);
  const changed = store.References.OfEntry((e: any) => e.relation(Paths.of(store.registered("crm.Knows"))).links({
    knower: (o: any) => o.path(Paths.of(ada)), known: (o: any) => o.path(Paths.of(cy)) }).properties({ since: 1999n }));
  assert(References.resolve(store, changed) === null); // a property no entry has
  const fewer = store.References.OfEntry((e: any) => e.relation(Paths.of(store.registered("crm.Knows"))).links({
    knower: (o: any) => o.path(Paths.of(ada)) }));
  assert(References.resolve(store, fewer) === null); // every link, or none
  assert(References.resolve(store, store.References.OfEntry((e: any) => e.relation(Paths.of(store.registered("crm.Knows")))))
    === null); // no links
}

## REF-05 · A frame a transaction gave back: each symbol's object, resolved to the frame

In [ ]:
const frame = frames[0];
{
  const reference = References.of(store, frame);
  assert(equal([...plain(reference).get("binds").keys()].sort(), [...frame.keys()].sort()));
  assert(equal(plain(reference).get("binds").get("ada"), plain(References.of(store, ada))));
  assert(equal(References.resolve(store, reference), frame));
  assert(equal(plain(References.of(store, new Map())), map({ binds: {} }))
    && equal(References.resolve(store, References.of(store, new Map())), new Map()));
  assert(equal(References.resolve(store, store.References.OfFrame((f: any) => f)), new Map())); // no binds: the empty frame
}

## REF-06 · What is refused: a counted object nothing names, another store's object, what is not an object, an entry or a frame

In [ ]:
{
  let loose: any;
  store.transaction(() => {
    loose = store.crm.Note((n: any) => n.text("by hand")); // counted, written outright: nothing names it
  });
  raises(LookupError, () => References.of(store, loose), "a counted 'crm.Note' has no reference: nothing names it");
  raises(LookupError, () => References.of(store, new Map([["loose", loose]])),
    "symbol 'loose' of the frame: a counted 'crm.Note' has no reference");
  raises(TypeError, () => References.of(other, ada), "the object belongs to another store");
  raises(TypeError, () => References.of(store, "x"), "References.of takes an object, a relation entry or a frame, not 'x'");
  raises(TypeError, () => References.of(store, new Map([["x", "x"]])),
    "References.of takes an object, a relation entry or a frame, not 'x'"); // a frame binding what is not an object
  store.transaction(() => {
    store.crm.Person(ada, (a: any) => a.trio((t: any) => t.insert((e: any) => e.b(bob).c(cy))));
  });
  raises(LookupError, () => References.of(store, [...ada.trio][0]), "has no path"); // an entry of a relation with no name of its own
  for (const wrong of ["x", Paths.of(ada)]) {
    raises(TypeError, () => References.resolve(store, wrong),
      "References.resolve takes a References.OfObject, OfEntry or OfFrame value of the store's");
  }
}

## REF-07 · A reference written and read back as a value; null where the store holds nothing it designates

In [ ]:
{
  const held = [...book.contacts][0]; // as the store holds it now: an update of ada's rewrote her entries
  for (const target of [ada, held, frame]) {
    const reference = References.of(store, target);
    const text = Json.ToJSON(store)(reference.as_schema(), reference);
    const back = Json.FromJSON(store)(reference.as_schema(), text);
    const found = References.resolve(store, back);
    assert(target instanceof Map ? equal(found, target) : found === target);
  }
  const empty: any = new B.Store(); // its schemas and nothing else
  empty.transaction(crm);
  for (const target of [ada, held, frame, Person]) {
    const reference = References.of(store, target);
    const back = Json.FromJSON(empty)(empty.registered(store.name_of(reference.as_schema())),
      Json.ToJSON(store)(reference.as_schema(), reference));
    assert(References.resolve(empty, back) === (target === Person ? empty.registered("crm.Person") : null));
  }
  const bare: any = new B.Store(); // not even the schemas
  const reference = References.of(store, held);
  assert(References.resolve(bare, Json.FromJSON(bare)(bare.References.OfEntry.as_schema(),
    Json.ToJSON(store)(reference.as_schema(), reference))) === null);
  const nowhere = store.References.OfObject((o: any) => o.path((p: any) => p.singleton(".").next((n: any) => n.entry(
    (e: any) => e.i("schemas").o("x").properties({ name: "crm" }).next((n: any) => n.target((t: any) => t))))));
  const gone = store.References.OfEntry((e: any) => e.relation(Paths.of(Person)).links({ knower: (o: any) => o.path(Paths.of(ada)) }));
  assert(References.resolve(store, nowhere) === null && References.resolve(store, gone) === null);
}

## REF-08 · A frame as a witness: property values an object holds, and relation entries, each by its objects' references

In [ ]:
{
  const witness = store.References.OfFrame((f: any) => f
    .binds({ ada: (o: any) => o.path(Paths.of(ada)), bob: (o: any) => o.path(Paths.of(bob)) })
    .assigns([(x: any) => x.as_object((p: any) => p.object((o: any) => o.path(Paths.of(ada))).properties({ name: "Ada" }))])
    .entries([(x: any) => x.as_object((e: any) => e.relation(Paths.of(store.registered("crm.Knows"))).links({
      knower: (o: any) => o.path(Paths.of(ada)), known: (o: any) => o.path(Paths.of(bob)) }).properties({ since: 2020n }))]));
  assert(equal(plain(witness).get("assigns"), [map({ object: plain(References.of(store, ada)), properties: { name: { str: "Ada" } } })]));
  assert(equal(plain(witness).get("entries"), [plain(References.of(store, [...ada.knows][0]))])); // an OfEntry, as of() gives one
  const text = Json.ToJSON(store)(witness.as_schema(), witness);
  assert(equal(plain(Json.FromJSON(store)(witness.as_schema(), text)), plain(witness))); // carried, and read back the same
  assert(equal(References.resolve(store, witness), new Map([["ada", ada], ["bob", bob]]))); // resolved: its bindings
}

## REF-09 · A pattern's symbol given a reference in place of a schema: it binds what the reference designates, one frame, or none; its spec filters or is written as any symbol's, by the schema its path implies where the store holds nothing it designates; a symbol has exactly one of a schema and a reference

In [ ]:
{
  const reference = References.of(store, ada); // a symbol given a reference binds what it designates, one frame each
  const known = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: reference, y: Person }).spec([(_: any) => _.x.knows([_.y])])]))]));
  assert(equal(known.map((f: any) => [f.get("x"), String(f.get("y").name)]), [[ada, "Bob"], [ada, "Cy"]])); // along an adjacency
  assert(equal(store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: References.of(store, bob), y: Person }).spec([(_: any) => _.x.knows([_.y])])]))]))
    .map((f: any) => f.get("y")), [cy]));
  assert(store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: reference }).spec([(_: any) => _.x.name("Bob")])]))])).length === 0); // its spec filters, as any symbol's
  const upserted = store.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ x: References.of(store, cy) }).spec([(_: any) => _.x.name("Cy C.")])]))]));
  assert(equal(upserted, [new Map([["x", cy]])]) && String(cy.name) === "Cy C."); // an upsert writes its spec over what it binds
  assert(equal(store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ s: References.of(store, Person) }).spec([(_: any) => _.s.name("crm.Person")])]))])), [new Map([["s", Person]])]));
  const nowhere = store.References.OfObject((o: any) => o.path((p: any) => p.singleton("crm.TheBook").next((n: any) => n.entry(
    (e: any) => e.i("contacts").o("book").properties({ key: "zed" }).next((n: any) => n.target((t: any) => t))))));
  for (const term of [(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols({ x: nowhere })])),
    (mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols({ x: nowhere })]))]) {
    assert(store.transaction((t: any) => t.mutations([term])).length === 0); // it designates nothing: no frame, nothing made
  }
  assert(store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: nowhere }).spec([(_: any) => _.x.name("Zed")])]))])).length === 0); // its spec, by the schema its path implies

  /** A `References.OfObject` of `held`'s, from a singleton through entries, each [i, o, keys]. */
  const at = (held: any, singleton: string, ...steps: [string, string, Record<string, unknown>][]): any => {
    const nested = (n: any, rest: [string, string, Record<string, unknown>][]): any => {
      if (rest.length === 0) return n.target((t: any) => t);
      const [[i, o, keys], ...more] = rest as [[string, string, Record<string, unknown>], ...[string, string, Record<string, unknown>][]];
      return n.entry((e: any) => e.i(i).o(o).properties(keys).next((n: any) => nested(n, more)));
    };
    return held.References.OfObject((o: any) => o.path((p: any) => p.singleton(singleton).next((n: any) => nested(n, steps))));
  };

  const client: any = new B.Store(); // the schemas, not the data: its spec is written by the schemas its reference's path goes through
  client.transaction(crm);
  const sent = client.Mutations.OfTransaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: at(client, "crm.TheBook", ["contacts", "book", { key: "ada" }]), y: client.registered("crm.Person") })
    .spec([(_: any) => _.x.knows([_.y])])]))]));
  const text = Json.dumps(Plain.ToPlain(client).Transaction(sent)); // written as JSON, read where the data is, and run
  assert(equal(store.transaction(Plain.FromPlain(store).Transaction(Json.loads(text))), known)); // the frames, as given above

  /** crm.Firm, owned through crm.Holds as a person is, by an adjacency also named `book`. */
  const firm = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("firm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Holds: S.OfRelation, Firm: S.OfObject }).spec([
        (_: any) => _.Holds.name("crm.Holds"),
        (_: any) => _.Firm.name("crm.Firm").ref(true).adjacencies([
          (t: any) => t.name("book").relation([_.Holds]).me("contact").owner(true)]),
      ])]))]);
  };

  const firms: any = new B.Store();
  firms.transaction(crm);
  firms.transaction(firm);
  for (const [held, given, why] of [
    [client, at(client, "crm.Nope"), "starts from 'crm.Nope', which no schema declares a singleton"],
    [client, at(client, "crm.TheBook", ["nope", "book", {}]),
      "steps through 'nope', an adjacency 'crm.Book' does not declare"],
    [client, at(client, "crm.TheBook", ["contacts", "nope", {}]), "steps through 'contacts' of 'crm.Book', where not "
      + "exactly one schema declares an adjacency 'nope' marked owner() on another link of its relation: found []"],
    [firms, at(firms, "crm.TheBook", ["contacts", "book", { key: "ada" }]), "steps through 'contacts' of 'crm.Book', "
      + "where not exactly one schema declares an adjacency 'book' marked owner() on another link of its relation: "
      + "found ['crm.Firm', 'crm.Person']"],
    [client, at(client, ".", ["schemas", "", { name: "Nope" }]), "names a schema object the store does not hold"],
  ] as [any, any, string][]) {
    const message = `symbol 'x' is given a reference whose path ${why}: a spec writes what its object's schema declares`;
    raises(LookupError, () => held.transaction(() => held.Schemas.OfPattern((p: any) => p.symbols({ x: given })
      .spec([(_: any) => _.x.name("Zed")]))), message);
  }
  let both: any, neither: any, given: any;
  store.transaction(() => {
    both = store.Schemas.OfPattern((p: any) => p.symbols([(s: any) => s.name("x").schema([Person]).reference(reference)]));
    neither = store.Schemas.OfPattern((p: any) => p.symbols([(s: any) => s.name("x")]));
    given = store.Schemas.OfPattern((p: any) => p.symbols({ x: reference }));
  });
  assert(equal(References.validate(both), ["symbol 'x' has both a schema and a reference: a symbol has exactly one"]));
  assert(equal(References.validate(neither), ["symbol 'x' has neither a schema nor a reference: a symbol has exactly one"]));
  assert(References.validate(given).length === 0 && given.symbols.get("x").schema === null);
  raises(TypeError, () => References.validate("x"), "References.validate takes a pattern, not 'x'");
  raises(TypeError, () => store.transaction(() => store.Schemas.OfPattern((p: any) => p.symbols([(s: any) => s.name("x")
    .schema([Person]).reference(reference)]).spec([(_: any) => _.x.name("Ada")]))),
    "symbol 'x' has both a schema and a reference: a symbol has exactly one");
}